In [1]:
import meshio
import numpy as np

In [6]:

m = meshio.read("../mesh/ref_01_2mm.inp")

tetra_cells = m.get_cells_type("tetra").copy()
p = m.points
print(f"Tetrahedra: {len(tetra_cells)}  |  Nodes: {len(p)}")

# ── Per-element Jacobian fix ───────────────────────────────────────────────
# Compute signed volume for every element; flip the two nodes of any inverted
# element individually. A global swap is wrong when Abaqus produces mixed
# handedness — that leaves an indefinite mass matrix and CG diverges.
a = p[tetra_cells[:, 0]]
b = p[tetra_cells[:, 1]]
c = p[tetra_cells[:, 2]]
d = p[tetra_cells[:, 3]]
vols = np.einsum("ij,ij->i", b - a, np.cross(c - a, d - a)) / 6.0

n_neg = (vols < 0).sum()
print(f"Inverted elements: {n_neg} / {len(tetra_cells)}")

# swap nodes 0 and 1 only for the inverted elements
mask = vols < 0
tetra_cells[mask, 0], tetra_cells[mask, 1] = (
    tetra_cells[mask, 1].copy(),
    tetra_cells[mask, 0].copy(),
)

# verify — all volumes should now be positive
a = p[tetra_cells[:, 0]]
b = p[tetra_cells[:, 1]]
c = p[tetra_cells[:, 2]]
d = p[tetra_cells[:, 3]]
vols_fixed = np.einsum("ij,ij->i", b - a, np.cross(c - a, d - a)) / 6.0
assert (vols_fixed > 0).all(), "Still inverted elements after fix!"
print(f"All volumes positive after fix ✓  (min={vols_fixed.min():.4g}  max={vols_fixed.max():.4g})")

# ── Write clean XDMF ──────────────────────────────────────────────────────
tetra_mesh = meshio.Mesh(
    points=p,
    cells=[("tetra", tetra_cells)],
)
out = "../mesh/ref_01_2mm.xdmf"
meshio.write(out, tetra_mesh)
print(f"Written → {out}")


Tetrahedra: 577220  |  Nodes: 111450
Inverted elements: 0 / 577220
All volumes positive after fix ✓  (min=0.1912  max=5.395)
Written → ../mesh/ref_01_2mm.xdmf
